In [1]:
code = 'SUT_TT'
pickle_path = 'C:/PICKLE/'
parameter_path = f'Parameter_{code}.csv'
meta_data_path = f"Parameter_{code}_MetaData.csv"
output_csv_path = f'{code}_output\\'

from pgcbacktest.BtParameters import *
from pgcbacktest.BacktestOptions import *

try:
    parameter, parameter_len = get_parameter_data(code, parameter_path)
    meta_data, meta_row_nos = get_meta_data(code, meta_data_path)
    os.makedirs(output_csv_path, exist_ok=True)
except Exception as e:
    input(str(e))

In [2]:
def SUT_TT(bt, start_time, end_time, orderside, sl, intra_sl, om, ut_orderside, ut_method, ut_sl, ut_om, tt_orderside, tt_method, tt_sl, tt_om):
    try:
        start_dt = datetime.datetime.combine(bt.current_date, start_time)
        end_dt = datetime.datetime.combine(bt.current_date, end_time)

        ce_scrip, pe_scrip, ce_price, pe_price, future_price, start_dt = bt.get_strike(start_dt, end_dt, om=om)
        if ce_scrip is None: return None

        entry_time = start_dt
        std_open, std_high, std_low, std_close, std_sl_price, std_intra_sl_price, std_sl_flag, std_intra_sl_flag, _, std_sl_time, std_pnl = bt.sl_check_combine_leg(start_dt, end_dt, ce_scrip, pe_scrip, sl=sl, intra_sl=intra_sl, orderside=orderside, with_ohlc=True)
        
        ut, ut_scrip, ut_open, ut_high, ut_low, ut_close, ut_sl_price, ut_sl_flag, ut_sl_time, ut_pnl = '', '', '', '', '', '', '', False, '', 0
        tt, tt_scrip, tt_open, tt_high, tt_low, tt_close, tt_sl_price, tt_sl_flag, tt_sl_time, tt_pnl = '', '', '', '', '', '', '', False, '', 0
        
        if std_sl_time and (std_sl_time < end_dt - datetime.timedelta(minutes=5)):
            
            ce_inc_rate = (bt.options_data.loc[(std_sl_time, ce_scrip), 'close'] - ce_price)/ce_price
            pe_inc_rate = (bt.options_data.loc[(std_sl_time, pe_scrip), 'close'] - pe_price)/pe_price
            
            if ce_inc_rate > pe_inc_rate:
                ut = 'PE' if ut_orderside == 'SELL' else 'CE'
                tt = 'CE' if tt_orderside == 'SELL' else 'PE' 
            elif ce_inc_rate < pe_inc_rate:
                ut = 'CE' if ut_orderside == 'SELL' else 'PE'
                tt = 'PE' if tt_orderside == 'SELL' else 'CE'

            if ut and tt:
                while std_sl_time < end_dt:
                    
                    ut_scrip, ut_price, ut_future_price, ut_start_dt = bt.get_strike(std_sl_time, std_sl_time+datetime.timedelta(minutes=1), om=ut_om, only=ut)
                    tt_scrip, tt_price, tt_future_price, tt_start_dt = bt.get_strike(std_sl_time, std_sl_time+datetime.timedelta(minutes=1), om=tt_om, only=tt)

                    if ut_scrip and tt_scrip and (ut_start_dt == tt_start_dt):
                        from_candle_close = True if ut_method == 'CC' else False
                        from_candle_close = True if tt_method == 'CC' else False
                        ut_open, ut_high, ut_low, ut_close, ut_sl_price, ut_sl_flag, _, _, ut_sl_time, ut_pnl = bt.sl_check_single_leg(ut_start_dt, end_dt, ut_scrip, sl=ut_sl, with_ohlc=True, orderside=ut_orderside, from_candle_close=from_candle_close)
                        tt_open, tt_high, tt_low, tt_close, tt_sl_price, tt_sl_flag, _, _, tt_sl_time, tt_pnl = bt.sl_check_single_leg(tt_start_dt, end_dt, tt_scrip, sl=tt_sl, with_ohlc=True, orderside=tt_orderside, from_candle_close=from_candle_close)
                        break
                    else:
                        std_sl_time += datetime.timedelta(minutes=1)
                        
        total_pnl = std_pnl + ut_pnl + tt_pnl
        return [code, bt.index, start_time, end_time, orderside, sl, intra_sl, om, ut_orderside, ut_method, ut_sl, ut_om, tt_orderside, tt_method, tt_sl, tt_om, bt.current_date.date(), bt.current_date.day_name(), bt.dte, entry_time.time(), future_price, ce_scrip, pe_scrip, std_open, std_high, std_low, std_close, std_sl_price, std_intra_sl_price, std_sl_flag, std_intra_sl_flag, std_sl_time, ut_scrip, ut_open, ut_high, ut_low, ut_close, ut_sl_price, ut_sl_flag, ut_sl_time, tt_scrip, tt_open, tt_high, tt_low, tt_close, tt_sl_price, tt_sl_flag, tt_sl_time, std_pnl, ut_pnl, tt_pnl, total_pnl]
    except Exception as e:
        print(e, [bt.index, bt.current_date, start_time, end_time, orderside, sl, intra_sl, om, ut_orderside, ut_method, ut_sl, ut_om, tt_orderside, tt_method, tt_sl, tt_om])
        return

In [3]:
for row_idx in range(len(meta_data)):
    
    if row_idx in meta_row_nos and meta_data.loc[row_idx, 'run']:
        try:
            meta_row = meta_data.iloc[row_idx]
            index, dte, from_date, to_date, start_time, end_time, date_lists = get_meta_row_data(meta_row, pickle_path)

            log_cols =('P_Strategy/P_Index/P_StartTime/P_EndTime/P_OrderSide/P_SL/P_intraSL/P_OM/P_UTOrderSide/P_UTMethod/P_UTSL/P_UTOM/P_TTOrderSide/P_TTMethod/P_TTSL/P_TTOM/Date/Day/DTE/EntryTime/Future/CE.Strike/PE.Strike/STD.Open/STD.High/STD.Low/STD.Close/STD.SL.Price/STD.IntraSL.Price/STD.SL.Flag/STD.IntraSL.Flag/STD.SL.Time/UT.Strike/UT.Open/UT.High/UT.Low/UT.Close/UT.SL.Price/UT.SL.Flag/UT.SL.Time/TT.Strike/TT.Open/TT.High/TT.Low/TT.Close/TT.SL.Price/TT.SL.Flag/TT.SL.Time/STD.PNL/UT.PNL/TT.PNL/Total.PNL').split('/')

            for current_date in date_lists:

                file_name = f"{index} {current_date.date()} {code}"
                if not is_file_exists(output_csv_path, file_name, parameter_len):

                    t1 = datetime.datetime.now()
                    print(f"Row-{row_idx} | File-{file_name} | Total-{parameter_len}")

                    bt = IntradayBacktest(pickle_path, index, current_date, dte, start_time, end_time)

                    for idx, i in enumerate(range(0, parameter_len, chunk_size), start=1):
                        chunck_file_name = f"{output_csv_path}{file_name} No-{idx}.parquet"
                        print(chunck_file_name)

                        chunk_parameter = parameter.iloc[i:i+chunk_size]
                        chunk = [SUT_TT(bt, row['entry_time'], row['exit_time'], row['orderside'], row['sl'], row['intra_sl'], row['om'], row['ut_orderside'], row['ut_method'], row['ut_sl'], row['ut_om'], row['tt_orderside'], row['tt_method'], row['tt_sl'], row['tt_om']) for idx, row in tqdm(chunk_parameter.iterrows(), total=len(chunk_parameter), colour='GREEN')]
                        save_chunk_data(chunk, log_cols, chunck_file_name)
                        
                        del chunk
                        del chunk_parameter
                        gc.collect()

                    del bt
                    gc.collect()
                    
                    t2 = datetime.datetime.now()
                    print(t2-t1)
        except Exception as e:
            input(str(e))

Row-0 | File-NIFTY 2022-01-06 SUT_TT | Total-1
SUT_TT_output\NIFTY 2022-01-06 SUT_TT No-1.parquet


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 72.66it/s]


0:00:00.145679
Row-0 | File-NIFTY 2022-01-13 SUT_TT | Total-1
SUT_TT_output\NIFTY 2022-01-13 SUT_TT No-1.parquet


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 89.76it/s]


0:00:00.131398
Row-0 | File-NIFTY 2022-01-20 SUT_TT | Total-1
SUT_TT_output\NIFTY 2022-01-20 SUT_TT No-1.parquet


...

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 66.73it/s]


0:00:00.157072
Row-0 | File-NIFTY 2022-06-02 SUT_TT | Total-1
SUT_TT_output\NIFTY 2022-06-02 SUT_TT No-1.parquet


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 78.48it/s]


0:00:00.151019
Row-0 | File-NIFTY 2022-06-09 SUT_TT | Total-1
SUT_TT_output\NIFTY 2022-06-09 SUT_TT No-1.parquet


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 62.70it/s]

KeyboardInterrupt

